# Imports and Configuration


In [1]:
# Cell 1 & 2 Combined and Fixed
import os
import torch
import random
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
import wandb

# --- Configuration ---






# 1. Path to your saved model weights
MODEL_CKPT_PATH = "./gemma3-141M-kinyarwanda-cpt-test-1/checkpoint-8000" 
TOKENIZER_PATH = "./gemma3-141M-kinyarwanda-cpt-test-1/final_model" 

DATA_PATH = "kinyarwanda_english_sft.csv"
OUTPUT_DIR = "./gemma3-141M-kinyarwanda-translator"

# --- Safety Check ---
if not os.path.exists(MODEL_CKPT_PATH):
    raise FileNotFoundError(f"🚨 I cannot find the folder: {MODEL_CKPT_PATH}. Check your spelling or current working directory!")

# Initialize WandB
wandb.init(
    project="gemma-3-kinyarwanda",
    name="sft-translation-chat-template",
    tags=["SFT", "Translation"]
)

print("Loading tokenizer and model...")

# Load tokenizer from the base path (where tokenizer.json lives)
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT_PATH, use_fast=True)
except OSError:
    print("Tokenizer not in checkpoint folder. Loading from base path instead...")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, use_fast=True)

# SFT requires right-padding
tokenizer.padding_side = "right" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model weights from the checkpoint path
model = AutoModelForCausalLM.from_pretrained(
    MODEL_CKPT_PATH, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
print("Model loaded successfully!")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/xmikelinux/.netrc.
wandb: Currently logged in as: almugabo to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`torch_dtype` is deprecated! Use `dtype` instead!


Loading tokenizer and model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded successfully!


## data formatting 

In [2]:

# Cell 3: Dataset Formatting
print("Loading and formatting dataset...")
dataset = load_dataset("csv", data_files=DATA_PATH, split="train")

# --- THE FIX: Manually set the Gemma Chat Template ---
gemma_template = "{% for message in messages %}{{ '<start_of_turn>' + message['role'] + '\n' + message['content'] + '<end_of_turn>\n' }}{% endfor %}"
tokenizer.chat_template = gemma_template
# ----------------------------------------------------

def format_chat_prompt(example):
    rw_text = str(example['kinyarwanda']).strip()
    en_text = str(example['english']).strip()
    
    # 50/50 chance to train En -> Rw or Rw -> En
    import random
    if random.random() < 0.5:
        user_prompt = f"please translate this english text into kinyarwanda\n\n{en_text}"
        model_response = rw_text
    else:
        user_prompt = f"hindura iyi nyandiko mu cyongereza\n\n{rw_text}"
        model_response = en_text
        
    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "model", "content": model_response}
    ]
    
    # Apply the template we just defined above!
    full_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )
    
    return {"text": full_prompt}

# Apply formatting and split
formatted_dataset = dataset.map(format_chat_prompt, remove_columns=dataset.column_names)
split_ds = formatted_dataset.train_test_split(test_size=0.05, seed=42)
train_data = split_ds["train"]
eval_data = split_ds["test"]

print(f"Training samples: {len(train_data)} | Validation samples: {len(eval_data)}")
print("\n--- Example Chat Prompt (Notice the <start_of_turn> tokens) ---")
print(train_data[0]["text"])

Loading and formatting dataset...
Training samples: 45421 | Validation samples: 2391

--- Example Chat Prompt (Notice the <start_of_turn> tokens) ---
<start_of_turn>user
hindura iyi nyandiko mu cyongereza

Hariho byinshi wakwigira kuri Mubera birenze guhura nawe<end_of_turn>
<start_of_turn>model
There is more to learn from Mubera than meeting her<end_of_turn>



##  Setup SFTTrainer

In [3]:
# Cell 4: Setup SFTTrainer
from trl import SFTConfig, SFTTrainer

print("Configuring SFT Trainer...")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,   # 3 is standard            
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4,
    learning_rate=5e-5,           
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,                    
    logging_steps=10,
    eval_strategy="epoch",        
    save_strategy="epoch",
    report_to="wandb",
    
    # --- UPDATED FOR NEW TRL VERSIONS ---
    dataset_text_field="text",    
    max_length=1024,                # <-- Changed from max_seq_length
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,           
    processing_class=tokenizer,    # <-- Changed from tokenizer
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Configuring SFT Trainer...


Truncating train dataset:   0%|          | 0/45421 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2391 [00:00<?, ? examples/s]

In [4]:
# Cell 5: Train!
print("Starting Supervised Fine-Tuning...")
#trainer.train()
trainer.train(resume_from_checkpoint=True)

trainer.save_model(f"{OUTPUT_DIR}/final_translator")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_translator")
wandb.finish()
print("🎉 SFT Complete! Your chat-templated translation model is ready.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 4}.
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Starting Supervised Fine-Tuning...


Epoch,Training Loss,Validation Loss
2,1.291425,1.402260
3,1.199259,1.415156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/entropy,█▁
eval/loss,▁█
eval/mean_token_accuracy,▁█
eval/num_tokens,▁█
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
train/entropy,█▆▇▇▅▇▇▇▅▅██▆▅▅▅▃▁▃▄▄▄▃▃▃▂▃▃▃▂▄▂▁▃▂▃▄▃▃▃
train/epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇███
+5,...


🎉 SFT Complete! Your chat-templated translation model is ready.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Configuration ---
# Point this to where your SFT model was saved. 
# If you stopped after 1 epoch, it might be in a checkpoint folder like:
# "./gemma3-141M-kinyarwanda-translator/checkpoint-XXXX"
# If it finished completely, use the final_translator folder:
MODEL_PATH = "./gemma3-141M-kinyarwanda-translator/final_translator"

print("Loading SFT Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
).eval() # Set to evaluation mode




Loading SFT Model and Tokenizer...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [12]:
# --- Translation Function ---
def translate(text, direction="en_to_rw"):
    """
    Translates text using the instruction-tuned model.
    direction: "en_to_rw" (English to Kinyarwanda) or "rw_to_en" (Kinyarwanda to English)
    """
    if direction == "en_to_rw":
        user_prompt = f"please translate this english text into kinyarwanda\n\n{text}"
    elif direction == "rw_to_en":
        user_prompt = f"hindura iyi nyandiko mu cyongereza\n\n{text}"
    else:
        raise ValueError("Direction must be 'en_to_rw' or 'rw_to_en'")

    messages = [{"role": "user", "content": user_prompt}]
    
    # ADDED return_dict=True
    inputs = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True 
    ).to(model.device)
    
    with torch.no_grad():
        # UNPACK inputs with ** 
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,       
            temperature=0.3,          
            do_sample=True,           
            pad_token_id=tokenizer.eos_token_id
        )
        
    # Extract ONLY the newly generated tokens using ["input_ids"]
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

In [13]:

# --- Let's Test It! ---
print("\n" + "="*50)
print("🌍 KINYARWANDA-ENGLISH TRANSLATOR READY 🌍")
print("="*50)

# Test 1: English to Kinyarwanda
en_text = "The family is traveling to the attractions."
print(f"\n[EN -> RW] Original: {en_text}")
print(f"Translation: {translate(en_text, direction='en_to_rw')}")

# Test 2: Kinyarwanda to English
rw_text = "Abantu baganira muburyo busanzwe."
print(f"\n[RW -> EN] Original: {rw_text}")
print(f"Translation: {translate(rw_text, direction='rw_to_en')}")

# Test 3: Try your own!
en_text_2 = "A group of young men standing at a table."
print(f"\n[EN -> RW] Original: {en_text_2}")
print(f"Translation: {translate(en_text_2, direction='en_to_rw')}")


🌍 KINYARWANDA-ENGLISH TRANSLATOR READY 🌍

[EN -> RW] Original: The family is traveling to the attractions.
Translation: model
Umuryango urimo kugenda ku miruho.

[RW -> EN] Original: Abantu baganira muburyo busanzwe.
Translation: model
People talk in general.

[EN -> RW] Original: A group of young men standing at a table.
Translation: model
Itsinda ryabagabo bato bahagaze kumeza.


In [14]:
xTxts = ["Perezida w’u Bufaransa, Emmanuel Macron, yatangaje ko igihe kigeze ngo igihugu cye cyongere intwaro kirimbuzi kibitse mu rwego rwo kwitegura gukumira uwagerageza kugihungabanyiriza umutekano no guhungabanya uw’inshuti z’abaturanyi" , 
         "Umunyamabanga wa Leta muri Minisiteri y’Ububanyi n’Amahanga, Usta Kayitesi, yakiriye kopi z’impapuro z’abadipolomate batatu bemejwe n’ibihugu byabo ngo babihagararire mu Rwanda.", 
         "Ni ibintu bikomeje guteza impaka abantu bibaza niba koko uyu mubyeyi yararisibye no mu mazina ye mu buryo bw’amategeko. Uyu muhungu mukuru wa Brad Pitt na Angelina Jolie ashobora kwiyongera kuri mushiki we, Shiloh, na we wakuweho izina rya se, na we agasiragana ‘Shiloh Jolie’ gusa." , 
         "Mani Martin yavuze ko uru rugendo rwamufashije kwiyubakamo imbaraga zo guharanira amahoro, ku rundi ruhande uyu muhanzi akaba yahise amenyesha abakunzi be ko bahise bamuha ikaze muri Hiroshima City University aho azataramira ihuriro ry’abanyeshuri bakomoka muri Afurika bigayo." 
        ]


In [15]:
for txt in xTxts:
    print(f"\n[RW -> EN] Original: {txt}")
    print(f"Translation: {translate(txt, direction='rw_to_en')}")
    print('----')


    


[RW -> EN] Original: Perezida w’u Bufaransa, Emmanuel Macron, yatangaje ko igihe kigeze ngo igihugu cye cyongere intwaro kirimbuzi kibitse mu rwego rwo kwitegura gukumira uwagerageza kugihungabanyiriza umutekano no guhungabanya uw’inshuti z’abaturanyi
Translation: model
French President Emmanuel Macron has said that the time has come for the country to rejoice in the war on nuclear weapons
----

[RW -> EN] Original: Umunyamabanga wa Leta muri Minisiteri y’Ububanyi n’Amahanga, Usta Kayitesi, yakiriye kopi z’impapuro z’abadipolomate batatu bemejwe n’ibihugu byabo ngo babihagararire mu Rwanda.
Translation: model
The secretary of State for Foreign Affairs, Usta Kayitesi, received three foreign diplomats at the end of their term in Rwanda.
----

[RW -> EN] Original: Ni ibintu bikomeje guteza impaka abantu bibaza niba koko uyu mubyeyi yararisibye no mu mazina ye mu buryo bw’amategeko. Uyu muhungu mukuru wa Brad Pitt na Angelina Jolie ashobora kwiyongera kuri mushiki we, Shiloh, na we wakuwe

In [ ]:
## Data 

In [ ]:
import trl

In [ ]:
import pandas as pd

# 1. Define the paths to your three TSV files

xFld = '/home/xmikelinux/Downloads/'

xfiles  = ['kinyarwanda-english-corpus2.tsv', 'kinyarwanda-english-corpus3.tsv', 'kinyarwanda-english-corpus.tsv']

file_paths = [xFld + x for x in xfiles]

# 2. Loop through and read each file
dataframes = []
for file in file_paths:
    # sep='\t' tells pandas it is a TSV. 
    # header=None means the first row is actual data, not column names.
    df = pd.read_csv(file, sep='\t', header=None, encoding='cp1252', names=['kinyarwanda', 'english'])
    dataframes.append(df)
    print(len(df))

# 3. Concatenate them all together vertically
combined_df = pd.concat(dataframes, ignore_index=True)

# 4. Clean the data (Drop any rows where a translation might be missing)
combined_df = combined_df.dropna()

# 5. Shuffle the dataset completely (frac=1 means 100% of the data)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 6. Save it to a clean, ready-to-use CSV
combined_df.to_csv('kinyarwanda_english_sft.csv', index=False)

print(len(combined_df))


print(f"✅ Successfully combined and shuffled {len(combined_df)} translation pairs!")
print("\nHere is a preview of your new dataset:")
print(combined_df.head())

In [ ]:
combined_df.head()

## TEST 